In [1]:
import os
from docx import Document
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

F:\Pytorch\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Architecture

# Folder Structure

# Template Structure

# LLM Client

In [2]:
try:
    import ollama
except ImportError:
    ollama = None


class LLMClient:
    """
    Generic LLM client with:
    - Ollama local testing mode
    - API-based production mode
    """

    def __init__(self, api_key=None, base_url=None, model=None, testing=True):

        self.api_key = api_key
        self.base_url = base_url.rstrip("/") if base_url else None
        self.model = model
        self.testing = testing

    # ----------------------------------------------------
    # MAIN ENTRY
    # ----------------------------------------------------

    def chat(self, prompt):

        if self.testing:
            return self._chat_ollama(prompt)

        return self._chat_api(prompt)

    # ----------------------------------------------------
    # LOCAL OLLAMA MODE
    # ----------------------------------------------------
    def generate(self, prompt):
        return self.chat(prompt)
    def _chat_ollama(self, prompt):

        if ollama is None:
            raise ImportError("ollama package not installed")

        response = ollama.chat(
            model=self.model or "llama3",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        return response["message"]["content"]

    # ----------------------------------------------------
    # API MODE (enterprise / internal LLM)
    # ----------------------------------------------------

    def _chat_api(self, prompt):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }

        response = requests.post(
            f"{self.base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"]

In [3]:
llm_client = LLMClient(
    testing=True,
    model="llama3"
)

## Step 1: Input arguments for:
* What type of GP
* Structure of GP

### 1.1 Input for GP

In [30]:
def detect_procedure_type(user_request):
    """
    Detects the operational procedure type from a user request.

    This function performs lightweight keyword-based classification
    to identify the intended industrial procedure category.

    Supported procedure types:
    - startup
    - shutdown
    - maintenance
    - inspection
    - general (fallback)

    Args:
        user_request (str):
            User query or procedure generation request.

    Returns:
        str:
            Detected procedure type.
    """

    text = user_request.lower()

    if "startup" in text:
        return "startup"
    if "shutdown" in text:
        return "shutdown"
    if "maintenance" in text:
        return "maintenance"
    if "inspection" in text:
        return "inspection"

    return "general"

In [31]:
user_input = detect_procedure_type("Tell me how to restart pressure")

### 1.2 Structure to GP

In [32]:
procedure_template = """
1. Objective
2. Scope
3. Preconditions
    3.1 Required Conditions
    3.2 Exceptions
4. Required Equipment
5. Safety Precautions
6. Procedure Steps
7. Validation Checks
8. Escalation Conditions
"""

## Step 2: Preparing documents for RAG

### Step 2.1 Parse logic from a word document

In [33]:
def parse_word_document(filepath):
    """
    Parses a .docx Word document into structured sections for downstream retrieval tasks.

    The function extracts content from a Word document and attempts to segment it into
    logical sections based on:

    1. Word heading styles (Heading 1, Heading 2, Swedish "Rubrik")
    2. Numeric text patterns (e.g., "2.1 Section Title")

    Each section is converted into a structured dictionary containing metadata and
    retrieval-ready text representations.

    If no structural signals (headings or numbered sections) are detected, the function
    falls back to creating a single section containing the first ~200 words of the
    document to ensure compatibility with downstream RAG pipelines.

    Processing steps:
    - Reads .docx file using python-docx
    - Detects headings via style metadata and regex patterns
    - Groups paragraph text into sections
    - Extracts section hierarchy (id, title, level)
    - Builds retrieval-friendly text fields for embeddings
    - Applies fallback chunking for unstructured documents

    Args:
        filepath (str):
            Path to the .docx file to be parsed.

    Returns:
        list[dict]:
            A list of section dictionaries. Each dictionary contains:
            - document (str): filename
            - section_id (str or None): hierarchical identifier (e.g., "2.1")
            - title (str): section title
            - level (int): heading depth level
            - raw_text (str): full section text
            - sentences (list[str]): sentence-level split of the section
            - section_text (str): formatted section text with header
            - retrieval_text (str): compact text optimized for embedding models

    Fallback behavior:
        If no structured sections are found, the function returns a single
        synthetic section built from the first ~200 words of the document.
    """

    doc = Document(filepath)

    sections = []

    current_section = None
    buffer = []

    has_any_heading = False  # 🔥 NEW FLAG

    def flush():
        nonlocal buffer, current_section

        if current_section and buffer:

            raw_text = "\n".join(buffer).strip()

            sections.append({
                "document": os.path.basename(filepath),

                "section_id": current_section["section_id"],
                "title": current_section["title"],
                "level": current_section["level"],

                "raw_text": raw_text,
                "sentences": buffer.copy(),

                "section_text": f"{current_section['section_id']} {current_section['title']}\n{raw_text}",
                "retrieval_text": f"{current_section['section_id']} {current_section['title']} {raw_text}"
            })

        buffer = []

    for para in doc.paragraphs:

        text = para.text.strip()
        if not text:
            continue

        style = para.style.name.lower()
        style = re.sub(r"\s+", " ", style)

        is_heading = (
            "heading 1" in style or "rubrik 1" in style or
            "heading 2" in style or "rubrik 2" in style
        )

        if is_heading:

            has_any_heading = True  # 🔥 mark that structure exists

            flush()

            match = re.match(r"^([\d\.]+)\s+(.*)", text)

            if match:
                section_id = match.group(1)
                title = match.group(2)
                level = section_id.count(".") + 1
            else:
                section_id = None
                title = text
                level = 1

            current_section = {
                "section_id": section_id,
                "title": title,
                "level": level
            }

        else:
            buffer.append(text)

    flush()

    # 🔥 SIMPLE FALLBACK: ensure at least one section exists
    if not sections:
        full_text = " ".join(
            p.text.strip()
            for p in doc.paragraphs
            if p.text.strip()
        )
    
        words = full_text.split()
    
        fallback_text = " ".join(words[:200])  # first 200 words
    
        sections = [{
            "document": os.path.basename(filepath),
            "section_id": "0",
            "title": "Full Document (Auto Chunk)",
            "level": 0,
            "raw_text": fallback_text,
            "sentences": fallback_text.split("."),
            "section_text": f"0 Full Document (Auto Chunk)\n{fallback_text}",
            "retrieval_text": f"0 Full Document (Auto Chunk) {fallback_text}"
        }]
    return sections

### Step 2.2 Load in documents

In [38]:
def load_all_documents(folder_path):
    """
    Loads and parses all .docx documents from a folder.

    This function:
    - Iterates through every .docx file in the provided folder
    - Parses each document using parse_word_document()
    - Aggregates all extracted sections into a single list

    Args:
        folder_path (str):
            Path to the folder containing Word documents.

    Returns:
        list[dict]:
            Combined list of parsed sections from all documents.
    """
    all_sections = []

    for file in os.listdir(folder_path):

        if file.endswith(".docx"):

            path = os.path.join(folder_path, file)
            sections = parse_word_document(path)

            all_sections.extend(sections)
    df = pd.DataFrame(all_sections)
    return all_sections, df

In [75]:
all_sections, df = load_all_documents("documents")


In [76]:
all_sections

[{'document': 'maintenance_guidelines.docx',
  'section_id': '0',
  'title': 'Full Document (Auto Chunk)',
  'level': 0,
  'raw_text': '1. Scope',
  'sentences': ['1', ' Scope'],
  'section_text': '0 Full Document (Auto Chunk)\n1. Scope',
  'retrieval_text': '0 Full Document (Auto Chunk) 1. Scope'},
 {'document': 'safety_manual.docx',
  'section_id': '0',
  'title': 'Full Document (Auto Chunk)',
  'level': 0,
  'raw_text': '1. Scope',
  'sentences': ['1', ' Scope'],
  'section_text': '0 Full Document (Auto Chunk)\n1. Scope',
  'retrieval_text': '0 Full Document (Auto Chunk) 1. Scope'},
 {'document': 'startup_manual.docx',
  'section_id': '0',
  'title': 'Full Document (Auto Chunk)',
  'level': 0,
  'raw_text': '1. Scope This document describes startup procedures for the PX-400 pressure system. 2. Startup Procedure 2.1 Initial Inspection Operators must inspect: - Pressure gauges - Valve seals - Cooling systems 2.2 System Activation Activate the primary control panel. Verify indicator li

### Step 2.3 First LLM to turn content into a structured format

In [77]:
def extract_requirements(section, llm_client):
    """
    Uses an LLM to extract explicit operational requirements from a section.

    This function:
    - Builds a structured prompt for requirement extraction
    - Sends the prompt to an LLM client
    - Parses the returned JSON response
    - Extracts operational requirements only
    - Attaches source metadata to each extracted requirement

    The extracted requirements include:
    - requirement_type
    - operational_phase
    - requirement_statement
    - criticality

    Args:
        section (dict):
            Structured document section.

        llm_client:
            LLM interface exposing a generate() method.

    Returns:
        list[dict]:
            Extracted requirement dictionaries.
    """


    prompt = f"""
You are an industrial operations analyst.

Extract ONLY explicit operational requirements from this SECTION.

SECTION:
{section['section_id']} {section['title']}

CONTENT:
{section['raw_text']}

Return JSON ONLY:
[
  {{
    "requirement_type": "...",
    "operational_phase": "...",
    "requirement_statement": "...",
    "criticality": "..."
  }}
]

Rules:
- No hallucinations
- No summarization
- Only explicit requirements
"""

    content = llm_client.generate(prompt)
    content = re.sub(r"```json|```", "", content)

    match = re.search(r"\[.*\]", content, re.DOTALL)
    if not match:
        return []

    try:
        reqs = json.loads(match.group(0))

        for r in reqs:
            r["source_document"] = section["document"]
            r["source_section_id"] = section["section_id"]
            r["source_title"] = section["title"]

        return reqs

    except:
        return []

### Step 2.4 Builds the requirements by using function from step 3

In [78]:
def build_requirement_database(sections, llm_client):
    """
    Builds a consolidated requirement database from document sections.

    This function:
    - Iterates through all parsed sections
    - Extracts operational requirements using extract_requirements()
    - Aggregates all extracted requirements into a single database

    Args:
        sections (list[dict]):
            Parsed document sections.

        llm_client:
            LLM interface used for requirement extraction.

    Returns:
        list[dict]:
            Combined requirement database.
    """

    all_reqs = []

    for s in sections:
        print(s)
        if not s["raw_text"].strip():
            continue

        reqs = extract_requirements(s, llm_client)
        all_reqs.extend(reqs)

    return all_reqs

In [79]:
generate_section

<function __main__.generate_section(section_name, sections, user_request, llm_client)>

In [80]:
all_reqs = build_requirement_database(all_sections, llm_client)

{'document': 'maintenance_guidelines.docx', 'section_id': '0', 'title': 'Full Document (Auto Chunk)', 'level': 0, 'raw_text': '1. Scope', 'sentences': ['1', ' Scope'], 'section_text': '0 Full Document (Auto Chunk)\n1. Scope', 'retrieval_text': '0 Full Document (Auto Chunk) 1. Scope'}
{'document': 'safety_manual.docx', 'section_id': '0', 'title': 'Full Document (Auto Chunk)', 'level': 0, 'raw_text': '1. Scope', 'sentences': ['1', ' Scope'], 'section_text': '0 Full Document (Auto Chunk)\n1. Scope', 'retrieval_text': '0 Full Document (Auto Chunk) 1. Scope'}
{'document': 'startup_manual.docx', 'section_id': '0', 'title': 'Full Document (Auto Chunk)', 'level': 0, 'raw_text': '1. Scope This document describes startup procedures for the PX-400 pressure system. 2. Startup Procedure 2.1 Initial Inspection Operators must inspect: - Pressure gauges - Valve seals - Cooling systems 2.2 System Activation Activate the primary control panel. Verify indicator lights are green. 2.3 Pressure Valve Releas

In [71]:
all_reqs

[]

### Step 2.5 Removes semantically similar text

In [81]:
def deduplicate_requirements(requirements, threshold=0.90):
    """
    Removes semantically duplicate requirements using embedding similarity.

    This function:
    - Encodes requirement statements into embeddings
    - Computes cosine similarity between requirements
    - Removes highly similar duplicate entries
    - Retains only unique operational requirements

    Args:
        requirements (list[dict]):
            Requirement dictionaries containing requirement statements.

        threshold (float, optional):
            Cosine similarity threshold for duplicate detection.
            Default is 0.90.

    Returns:
        list[dict]:
            Deduplicated list of requirements.
    """

    if not requirements:
        return []

    texts = [r["requirement_statement"] for r in requirements]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True
    )

    unique = []
    used = set()

    for i in range(len(requirements)):

        if i in used:
            continue

        unique.append(requirements[i])

        for j in range(i + 1, len(requirements)):

            sim = cosine_similarity(
                [embeddings[i]],
                [embeddings[j]]
            )[0][0]

            if sim >= threshold:
                used.add(j)

    return unique

In [82]:
requirements = deduplicate_requirements(all_reqs)

### Step 2.6 Build retrieval index

In [87]:
def build_requirement_index(sections, requirements):
    """
    Builds a hybrid retrieval index for section-level RAG retrieval.

    This function creates the core retrieval system used for
    section-grounded SOP generation. The primary retrieval unit
    is the full document section rather than fragmented requirement
    statements.

    The retrieval index combines:
    - Semantic embeddings for vector similarity search
    - TF-IDF lexical search for keyword matching
    - Requirement traceability mappings for validation and auditing

    Processing steps:
    1. Extracts retrieval-ready section text
    2. Generates normalized semantic embeddings
    3. Builds a requirement lookup table grouped by section ID
    4. Creates a TF-IDF matrix for hybrid lexical retrieval
    5. Returns a consolidated retrieval system dictionary

    Args:
        sections (list[dict]):
            Parsed document sections containing section metadata
            and retrieval text.

        requirements (list[dict]):
            Extracted operational requirements linked to source sections.

    Returns:
        dict:
            Retrieval system containing:
            - sections
            - section_texts
            - section_embeddings
            - requirements
            - requirement_lookup
            - tfidf vectorizer
            - tfidf matrix

    Raises:
        ValueError:
            If no sections are provided.
    """

    if not sections:
        raise ValueError("No sections found")

    ########################################################
    # 🔥 CORE IDEA: SECTION = RETRIEVAL UNIT
    ########################################################

    section_texts = [
        s["section_text"]
        for s in sections
    ]

    section_embeddings = embedding_model.encode(
        section_texts,
        normalize_embeddings=True
    )

    ########################################################
    # REQUIREMENTS ONLY FOR TRACEABILITY / VALIDATION
    ########################################################

    requirement_lookup = {}

    for r in requirements:
        sid = r.get("source_section_id")
        requirement_lookup.setdefault(sid, []).append(r)

    ########################################################
    # HYBRID LEXICAL INDEX (SECTION LEVEL)
    ########################################################

    tfidf = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf_matrix = tfidf.fit_transform(section_texts)

    return {
        ####################################################
        # PRIMARY RETRIEVAL UNIT
        ####################################################
        "sections": sections,
        "section_texts": section_texts,
        "section_embeddings": np.array(section_embeddings),

        ####################################################
        # TRACEABILITY LAYER
        ####################################################
        "requirements": requirements,
        "requirement_lookup": requirement_lookup,

        ####################################################
        # HYBRID SEARCH LAYER
        ####################################################
        "tfidf": tfidf,
        "tfidf_matrix": tfidf_matrix
    }

In [84]:
alls = build_requirement_index(all_sections, requirements)

In [ ]:
alls

## Step 3 Retrieve info using RAG

### Step 3.1 Retrieve Requirements using RAG

In [18]:
def retrieve_requirements(
    query,
    retrieval_system,
    procedure_type,
    requirement_type=None,
    k=5,
    alpha=0.6
):
    """
    Retrieves the most relevant document sections using hybrid RAG retrieval.

    This function implements section-level retrieval using a hybrid
    semantic + lexical search strategy.

    Retrieval process:
    1. Encodes the query into semantic embeddings
    2. Computes cosine similarity against section embeddings
    3. Performs TF-IDF lexical similarity search
    4. Normalizes semantic and lexical scores
    5. Combines scores using weighted hybrid ranking
    6. Returns the top-k most relevant sections

    Core design principle:
    - Entire document sections are treated as the retrieval unit
      rather than isolated requirement fragments.

    Args:
        query (str):
            User query or SOP generation request.

        retrieval_system (dict):
            Hybrid retrieval index generated by build_requirement_index().

        procedure_type (str):
            Detected procedure category.

        requirement_type (str, optional):
            Optional requirement category filter.

        k (int, optional):
            Number of top sections to retrieve.
            Default is 5.

        alpha (float, optional):
            Weighting factor between semantic and lexical retrieval.
            - alpha = 1.0 → semantic only
            - alpha = 0.0 → lexical only
            Default is 0.6.

    Returns:
        list[dict]:
            Top-k retrieved document sections ranked by hybrid relevance.
    """


    ########################################################
    # 🔥 PRIMARY RETRIEVAL UNIT = SECTIONS
    ########################################################

    sections = retrieval_system["sections"]

    section_embeddings = retrieval_system["section_embeddings"]

    ########################################################
    # SEMANTIC SEARCH (SECTION LEVEL)
    ########################################################

    query_emb = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(query_emb, section_embeddings)[0]

    ########################################################
    # LEXICAL SEARCH (SECTION LEVEL)
    ########################################################

    tfidf_vec = retrieval_system["tfidf"].transform([query])

    lexical_scores = cosine_similarity(
        tfidf_vec,
        retrieval_system["tfidf_matrix"]
    )[0]

    ########################################################
    # NORMALIZATION
    ########################################################

    def norm(x):
        if x.max() == x.min():
            return x * 0
        return (x - x.min()) / (x.max() - x.min())

    semantic_scores = norm(semantic_scores)
    lexical_scores = norm(lexical_scores)

    combined_scores = alpha * semantic_scores + (1 - alpha) * lexical_scores

    ########################################################
    # TOP-K SECTIONS
    ########################################################

    top_idx = np.argsort(combined_scores)[::-1][:k]

    return [sections[i] for i in top_idx]


### Step 3.2 Formats the retrieved RAG info

In [88]:
def format_requirements(requirements):
    """
    Formats retrieved requirements into a structured text representation.

    This function converts requirement dictionaries into a standardized,
    human-readable text format suitable for prompt grounding in LLM-based
    SOP generation.

    Included fields:
    - requirement type
    - operational phase
    - criticality
    - requirement statement
    - source traceability information

    Args:
        requirements (list[dict]):
            Retrieved requirement dictionaries.

    Returns:
        str:
            Formatted multiline requirement text block.
    """

    out = ""

    for i, r in enumerate(requirements):

        out += f"""
Requirement {i+1}
Type: {r.get('requirement_type', 'unknown')}
Phase: {r.get('operational_phase', 'unknown')}
Criticality: {r.get('criticality', 'unknown')}
Statement: {r.get('requirement_statement', 'unknown')}
Source: {r.get('source_section_id', 'UNMAPPED')} | {r.get('source_title', 'unknown')}
"""

    return out

## Step 4: Generate section with the help of RAG

### Step 4.1 Generates the text

In [91]:
def generate_section(section_name, sections, user_request, llm_client):
    """
    Generates an General Procedure section using fully grounded section-based retrieval.

    This function:
    - Formats retrieved sections into grounding context
    - Constructs a constrained generation prompt
    - Enforces strict retrieval-grounded generation rules
    - Uses an LLM to generate SOP content without hallucination

    The generation process is designed to:
    - Use ONLY retrieved source material
    - Prevent invention of missing operational steps
    - Preserve procedural ordering and structure
    - Maintain traceable grounding to source documentation

    Prompt constraints explicitly prohibit:
    - Hallucinations
    - General knowledge additions
    - Summarization
    - Meta-commentary
    - Unsupported procedural steps

    Args:
        section_name (str):
            Target SOP section to generate.

        sections (list[dict]):
            Retrieved sections used as grounding context.

        user_request (str):
            Original user request or procedural objective.

        llm_client:
            LLM interface exposing a generate() method.

    Returns:
        str:
            Generated SOP section content.
    """

    section_text = format_requirements(sections)

    prompt = f"""
You are generating an General Procedure section.

This system uses FULL SECTION-BASED RETRIEVAL (not fragmented requirements).

############################################################
USER CONTEXT
############################################################
User Request:
{user_request}

Target Section:
{section_name}

############################################################
RETRIEVED SECTIONS (GROUND TRUTH)
############################################################
{section_text}

############################################################
STRICT RULES (HARD CONSTRAINTS)
############################################################

- Use ONLY retrieved section content
- Do NOT invent missing steps
- Do NOT add general knowledge
- Do NOT explain or summarize
- Do NOT repeat metadata (IDs, titles, sources)
- DO NOT include meta-commentary like “Note:”
- DO NOT explain what you are doing

############################################################
OUTPUT RULES
############################################################

- Write ONLY section body
- Maintain SOP structure and flow
- Use bullet points where appropriate
- Preserve ordering implied in sections
- Ensure full grounding in retrieved sections

############################################################
HARD CONSTRAINT
############################################################

If something is not in retrieved sections:
➡ DO NOT include it

Return ONLY the section content.
"""

    return llm_client.generate(prompt)

### 4.2 Coordinater function dictating structure to the above function

In [ ]:
def generate_general_procedure(
    user_request,
    retrieval_system,
    llm_client,
    procedure_template
):
    """
    Generates a complete General Procedure using section-based RAG retrieval.

    This function orchestrates section-level procedural generation by:
    - Detecting the requested procedure type
    - Parsing the provided SOP template structure
    - Mapping template headings to operational requirement categories
    - Retrieving the most relevant document sections for each SOP section
    - Generating grounded procedural content using retrieved sections
    - Constructing the final SOP in Markdown format

    Core architectural principle:
    - Full document sections are treated as the primary retrieval unit
      instead of isolated requirement fragments.

    Generation workflow:
    1. Detect procedure type from user request
    2. Parse template headings
    3. Infer section intent (safety, startup, shutdown, etc.)
    4. Build section-specific retrieval queries
    5. Retrieve relevant sections using hybrid RAG search
    6. Generate grounded SOP content for each section
    7. Assemble final structured procedure document

    The function supports hierarchical SOP structures:
    - Level 1 → ##
    - Level 2 → ###
    - Level 3+ → ####

    Supported procedural section mappings include:
    - safety
    - validation
    - equipment
    - escalation
    - operational
    - startup
    - shutdown
    - inspection
    - maintenance

    Args:
        user_request (str):
            User-defined procedural objective or request.

        retrieval_system (dict):
            Section-RAG retrieval system generated by
            build_requirement_index().

        llm_client:
            LLM interface exposing a generate() method.

        procedure_template (str):
            Structured SOP template defining procedural sections.

    Returns:
        str:
            Fully generated Markdown-formatted General Procedure.
    """

    procedure_type = detect_procedure_type(user_request)

    template_lines = [
        line.strip()
        for line in procedure_template.splitlines()
        if line.strip()
    ]

    generated_sections = []

    section_type_mapping = {
        "safety": "safety",
        "validation": "validation",
        "equipment": "equipment",
        "escalation": "escalation",
        "precondition": "operational",
        "procedure": "operational",
        "startup": "startup",
        "shutdown": "shutdown",
        "inspection": "inspection",
        "maintenance": "maintenance"
    }

    for line in template_lines:

        clean_heading = re.sub(r"^\d+(\.\d+)*\s*", "", line).strip()
        lower_heading = clean_heading.lower()

        req_type = None
        for keyword, mapped_type in section_type_mapping.items():
            if keyword in lower_heading:
                req_type = mapped_type
                break

        ####################################################
        # 🔥 SECTION-RAG QUERY (STRONG SIGNAL)
        ####################################################

        query = (
            f"Procedure type: {procedure_type}. "
            f"User intent: {user_request}. "
            f"Section: {clean_heading}"
        )

        sections = retrieve_requirements(
            query=query,
            retrieval_system=retrieval_system,
            procedure_type=procedure_type,
            requirement_type=req_type,
            k=6
        )

        section_content = generate_section(
            section_name=clean_heading,
            sections=sections,
            user_request=user_request,
            llm_client=llm_client
        )

        level = line.count(".") + 1

        if level <= 1:
            markdown_heading = f"## {clean_heading}"
        elif level == 2:
            markdown_heading = f"### {clean_heading}"
        else:
            markdown_heading = f"#### {clean_heading}"

        generated_sections.append(
            f"{markdown_heading}\n\n{section_content}"
        )

    return "# General Procedure\n\n" + "\n\n".join(generated_sections)

## Step 5: Validates the generated text

### Step 5.1 Checks if requirements where included

In [92]:
def validate_requirement_coverage(
    procedure_text,
    requirements,
    llm_client
):
    """
    Validates requirement coverage within a generated General Procedure.

    This function performs requirement-level traceability validation
    against a generated procedure using an LLM-based evaluator.

    Validation process:
    - Compares generated SOP content against extracted requirements
    - Evaluates whether each requirement is:
        1. Included
        2. Partially Included
        3. Missing
    - Preserves traceability back to original source sections
    - Enforces strict evaluation using only provided procedure text

    The validation is designed for section-based RAG systems where:
    - Full sections are the primary retrieval unit
    - Requirements serve as compliance anchors
    - Structural fidelity is prioritized over inferred knowledge

    Prompt constraints explicitly prohibit:
    - External assumptions
    - Hallucinated standards
    - Inferred operational knowledge
    - Requirement invention

    Args:
        procedure_text (str):
            Generated SOP or procedural content.

        requirements (list[dict]):
            Extracted operational requirements used for validation.

        llm_client:
            LLM interface exposing a generate() method.

    Returns:
        str:
            JSON-formatted validation results containing:
            - requirement statement
            - coverage status
            - validation reasoning
            - source traceability metadata
    """

    requirement_text = format_requirements(requirements)

    prompt = f"""
You are validating an General Procedure generated using FULL SECTION-BASED RETRIEVAL.

SYSTEM CONTEXT:
- The system retrieves full sections (not fragments)
- Requirements are derived from sections for traceability only
- The SOP is structured by section blocks

############################################################
REQUIREMENTS (TRACEABILITY LAYER ONLY)
############################################################
{requirement_text}

############################################################
GENERATED PROCEDURE
############################################################
{procedure_text}

############################################################
TASK
############################################################

For EACH requirement classify:

1. Included (fully satisfied)
2. Partially Included (referenced but incomplete)
3. Missing (not represented)

############################################################
RULES
############################################################

- Evaluate ONLY against procedure text
- Do NOT assume missing knowledge exists
- Do NOT introduce external standards
- Be strict but realistic for industrial SOPs
- Respect section-based structure

############################################################
OUTPUT FORMAT (STRICT JSON ONLY)
############################################################

[
  {{
    "requirement_statement": "...",
    "status": "Included | Partially Included | Missing",
    "reason": "...",
    "source_section_id": "...",
    "source_title": "..."
  }}
]
"""

    return llm_client.generate(prompt)

### Step 5.2 Peforms overall validation

In [90]:
def validate_procedure(
    procedure_text,
    requirements,
    llm_client
):
    """
    Performs enterprise-level validation of a generated General Procedure.

    This function executes comprehensive structural, procedural, General Procedure
    and compliance-focused validation using a section-grounded
    validation framework.

    The validator assumes:
    - Full document sections are the source of truth
    - Requirements provide traceability only
    - Generated procedures must remain fully grounded in retrieved sections

    Validation categories include:
    1. Unsupported Statements
       - Detects content not grounded in retrieved sections or requirements

    2. Missing Coverage
       - Identifies missing procedural areas implied by requirements

    3. Ambiguity / Non-Executable Language
       - Detects vague or operationally unclear instructions

    4. Procedural Flow Consistency
       - Validates lifecycle ordering:
         startup → operation → shutdown → maintenance

    5. Cross-Section Consistency
       - Detects contradictory instructions across sections

    6. Provenance Alignment Check
       - Ensures requirements appear in the correct procedural context

    7. Structural Integrity Check
       - Detects missing or collapsed SOP sections

    8. Overall Assessment
       - Produces an overall procedural risk rating

    Prompt constraints explicitly prohibit:
    - External engineering assumptions
    - Hallucinated requirements
    - Invented operational logic
    - Unsupported procedural inference

    Args:
        procedure_text (str):
            Generated SOP content.

        requirements (list[dict]):
            Requirement traceability layer used for validation.

        llm_client:
            LLM interface exposing a generate() method.

    Returns:
        str:
            JSON-formatted enterprise validation report containing:
            - unsupported statements
            - missing coverage
            - ambiguities
            - flow issues
            - cross-section inconsistencies
            - provenance mismatches
            - structural issues
            - overall assessment
    """

    requirement_text = format_requirements(requirements)

    prompt = f"""
You are a senior industrial compliance reviewer.

This procedure is generated using a SECTION-BASED RETRIEVAL SYSTEM:

- Retrieval unit = full document sections (not isolated facts)
- Requirements = traceability and compliance anchors only
- Procedure = structured SOP derived from sections
- System priority = structural correctness over detail inference

############################################################
GROUND TRUTH REQUIREMENTS (TRACEABILITY LAYER)
############################################################
{requirement_text}

############################################################
GENERATED PROCEDURE
############################################################
{procedure_text}

############################################################
CRITICAL VALIDATION RULES (HARD CONSTRAINTS)
############################################################

- You MUST treat sections as the original structural truth
- You MUST NOT assume missing operational knowledge
- You MUST NOT introduce external engineering logic
- You MUST NOT invent missing requirements or steps
- If something is missing → explicitly mark it as missing
- Be strict, but only within provided context
- You MUST NOT create requirement IDs or references that are not explicitly present in the input.


############################################################
SECTION-AWARE VALIDATION TASKS
############################################################

1. Unsupported Statements
   - Identify statements NOT grounded in:
     a) provided requirements OR
     b) clear section-derived intent
   - Quote exact offending text

2. Missing Coverage
   - Identify missing procedural areas (e.g., safety, startup, shutdown steps)
   - Only infer gaps from requirement set (no external assumptions)

3. Ambiguity / Non-Executable Language
   - Identify instructions that cannot be executed operationally
   - Focus on:
     * missing thresholds
     * undefined terms
     * vague verbs (e.g. "ensure", "properly", "as needed")

4. Procedural Flow Consistency
   - Validate correct lifecycle order:
     startup → operation → shutdown → maintenance
   - Detect ordering issues or missing phases

5. Cross-Section Consistency
   - Identify contradictions across procedure sections
   - Focus on conflicting instructions between phases

6. Provenance Alignment Check
   - Safety requirements must appear in safety-related sections
   - Equipment requirements must appear in correct operational context
   - Misplacement must be flagged

7. Structural Integrity Check
   - Ensure all major SOP sections implied by structure are present
   - Detect missing or collapsed sections

8. Overall Assessment
   - Rate: Low / Medium / High
   - Based ONLY on provided inputs (no external knowledge)

############################################################
OUTPUT FORMAT (STRICT JSON ONLY)
############################################################

{{
  "unsupported_statements": [
    {{
      "statement": "",
      "reason": ""
    }}
  ],
  "missing_coverage": [
    {{
      "gap": "",
      "evidence_from_requirements": ""
    }}
  ],
  "ambiguities": [
    {{
      "statement": "",
      "issue": ""
    }}
  ],
  "flow_issues": [
    {{
      "issue": "",
      "location": ""
    }}
  ],
  "cross_section_issues": [
    {{
      "issue": "",
      "conflict": ""
    }}
  ],
  "provenance_mismatches": [
    {{
      "issue": "",
      "expected_section_type": "",
      "observed_location": ""
    }}
  ],
  "structural_issues": [
    {{
      "issue": "",
      "missing_section": ""
    }}
  ],
  "overall_assessment": {{
    "rating": "Low | Medium | High",
    "summary": ""
  }}
}}

IMPORTANT:
- Return ONLY valid JSON
- No explanations
- No commentary
- No additional text
"""
    
    return llm_client.generate(prompt)


## Step 6: Save doc file

In [23]:
def save_procedure_docx(
    procedure_text,
    validation_text,
    output_path="output/generated_procedure.docx",
    output_path2="output/validation.docx"
):
    """
    Saves generated SOP and validation results as Word documents.

    This function creates two .docx files:
    1. Generated procedure document
    2. Validation report document

    Procedure document formatting:
    - Converts Markdown-style headings into Word headings
    - Converts bullet syntax into Word bullet lists
    - Preserves paragraph structure for SOP readability

    Validation document formatting:
    - Attempts to parse validation output as JSON
    - Pretty-prints structured validation results
    - Falls back to plain text formatting if JSON parsing fails

    Supported heading formats:
    - # Heading 1
    - ## Heading 2
    - ### Heading 3
    - #### Heading 4

    Supported bullet formats:
    - "- item"
    - "* item"

    Args:
        procedure_text (str):
            Generated SOP content to save.

        validation_text (str):
            Validation results generated by the validation pipeline.

        output_path (str, optional):
            File path for the generated procedure document.
            Default is:
            "output/generated_procedure.docx"

        output_path2 (str, optional):
            File path for the validation document.
            Default is:
            "output/validation.docx"

    Returns:
        tuple[str, str]:
            Paths to the saved procedure and validation documents.
    """

    ########################################################
    # PROCEDURE DOCUMENT
    ########################################################

    doc = Document()

    lines = procedure_text.split("\n")

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        # HEADINGS
        if stripped.startswith("#### "):
            doc.add_heading(stripped[5:], level=4)

        elif stripped.startswith("### "):
            doc.add_heading(stripped[4:], level=3)

        elif stripped.startswith("## "):
            doc.add_heading(stripped[3:], level=2)

        elif stripped.startswith("# "):
            doc.add_heading(stripped[2:], level=1)

        # BULLETS
        elif stripped.startswith("- ") or stripped.startswith("* "):
            doc.add_paragraph(stripped[2:], style="List Bullet")

        else:
            doc.add_paragraph(stripped)

    doc.save(output_path)

    ########################################################
    # VALIDATION DOCUMENT
    ########################################################

    doc2 = Document()
    doc2.add_heading("Validation Results", level=1)

    # Try structured formatting first
    try:
        parsed = json.loads(validation_text)
        pretty = json.dumps(parsed, indent=2)

        for line in pretty.split("\n"):
            doc2.add_paragraph(line)

    except Exception:
        for line in validation_text.split("\n"):
            if line.strip():
                doc2.add_paragraph(line)

    doc2.save(output_path2)

    return output_path, output_path2

## Run the workflow

In [26]:
def run_workflow(
    user_request,
    document_folder,
    llm_client,
    procedure_template,
    progress_callback=None
):
    """
    Executes the complete end-to-end SECTION-RAG SOP generation workflow.

    This function serves as the main orchestration pipeline for:
    - document ingestion
    - requirement extraction
    - retrieval index construction
    - grounded SOP generation
    - validation
    - document export
    - provenance tracking
    - workflow analytics

    The workflow is built around a SECTION-RAG architecture where:
    - Full document sections are the primary retrieval unit
    - Requirements provide traceability and compliance anchoring
    - SOP generation is fully grounded in retrieved sections

    Workflow stages:
    --------------------------------------------------------
    STEP 1 — Load Sections
        - Parse Word documents into structured sections

    STEP 2 — Requirement Extraction
        - Extract operational requirements using LLM analysis

    STEP 3 — Requirement Deduplication
        - Remove semantically duplicate requirements

    STEP 4 — Build Retrieval Index
        - Create hybrid semantic + lexical retrieval system

    STEP 5 — Procedure Generation
        - Generate grounded SOP sections using SECTION-RAG

    STEP 6 — Requirement Coverage Validation
        - Validate requirement inclusion and traceability

    STEP 7 — Enterprise Validation
        - Perform structural and compliance validation

    STEP 8 — Save Outputs
        - Export SOP and validation results to Word documents

    Additional system capabilities:
    - Runtime timing analytics
    - Progress callback support
    - Provenance tracking
    - Retrieval transparency layer
    - Requirement traceability mapping

    Args:
        user_request (str):
            User-defined SOP generation request.

        document_folder (str):
            Path to folder containing source .docx documents.

        llm_client:
            LLM interface exposing a generate() method.

        procedure_template (str):
            SOP template defining target document structure.

        progress_callback (callable, optional):
            Optional callback function for workflow progress updates.
            Expected signature:
                callback(step_description, percent_complete)

    Returns:
        dict:
            Complete workflow result package containing:

            Generated Artifacts:
            - procedure
            - coverage validation
            - enterprise validation
            - output document paths

            Core Data:
            - extracted requirements
            - procedure template

            Analytics:
            - workflow timing metrics
            - total runtime

            Transparency / Provenance:
            - retrieval sources
            - provenance mappings
            - loaded document metadata

            System Statistics:
            - documents loaded
            - total requirements
            - runtime seconds
    """

    start_time = time.time()

    step_timings = {}
    step_start = None
    last_step = None

    ########################################################
    # PROGRESS UPDATE
    ########################################################

    def update(step, percent):
        if progress_callback:
            progress_callback(step, percent)
        else:
            print(step)

    ########################################################
    # TIMING TRACKER
    ########################################################

    def mark_step(name):
        nonlocal step_start, last_step

        if step_start is not None and last_step is not None:
            step_timings[last_step] = round(
                time.time() - step_start,
                2
            )

        step_start = time.time()
        last_step = name

    ########################################################
    # STEP 1 — LOAD SECTIONS (TRUE SECTION-RAG INPUT)
    ########################################################

    mark_step("Loading documents")
    update("STEP 1 — Loading documents", 5)

    sections = load_all_documents(document_folder)

    update(f"Loaded sections: {len(sections)}", 10)

    ########################################################
    # SECTION-LEVEL PROVENANCE (SOURCE TRUTH LAYER)
    ########################################################

    retrieval_sources = [
        {
            "document": s.get("document", "unknown"),
            "section_id": s.get("section_id"),
            "title": s.get("title"),
            "level": s.get("level"),
            "retrieval_text": s.get("retrieval_text", "")
        }
        for s in sections
    ]

    ########################################################
    # STEP 2 — REQUIREMENT EXTRACTION (TRACEABILITY LAYER)
    ########################################################

    mark_step("Extracting requirements")
    update("STEP 2 — Extracting requirements", 20)

    requirements = build_requirement_database(sections, llm_client)

    update(f"Extracted requirements: {len(requirements)}", 30)

    ########################################################
    # STEP 3 — DEDUPLICATION
    ########################################################

    mark_step("Deduplicating requirements")
    update("STEP 3 — Deduplicating requirements", 40)

    requirements = deduplicate_requirements(requirements)

    update(f"Unique requirements: {len(requirements)}", 50)

    ########################################################
    # STEP 4 — BUILD SECTION-RAG INDEX
    ########################################################

    mark_step("Building retrieval system")
    update("STEP 4 — Building retrieval system", 60)

    retrieval_system = build_requirement_index(sections, requirements)

    ########################################################
    # PROVENANCE + TRACEABILITY LAYER (FINAL STRUCTURE)
    ########################################################

    retrieval_system["sources"] = retrieval_sources

    retrieval_system["documents"] = sorted({
        s.get("document", "unknown")
        for s in sections
    })

    retrieval_system["provenance_map"] = [
        {
            "requirement_statement": r["requirement_statement"],
            "document": r.get("source_document"),
            "section_id": r.get("source_section_id"),
            "title": r.get("source_title"),
            "criticality": r.get("criticality"),
            "requirement_type": r.get("requirement_type"),
            "operational_phase": r.get("operational_phase")
        }
        for r in requirements
    ]

    ########################################################
    # STEP 5 — PROCEDURE GENERATION (SECTION-RAG CORE)
    ########################################################

    mark_step("Generating procedure")
    update("STEP 5 — Generating procedure", 70)

    procedure = generate_general_procedure(
        user_request=user_request,
        retrieval_system=retrieval_system,
        llm_client=llm_client,
        procedure_template=procedure_template
    )

    ########################################################
    # STEP 6 — COVERAGE VALIDATION
    ########################################################

    mark_step("Requirement coverage validation")
    update("STEP 6 — Requirement coverage validation", 82)

    coverage = validate_requirement_coverage(
        procedure,
        requirements,
        llm_client
    )

    ########################################################
    # STEP 7 — ENTERPRISE VALIDATION
    ########################################################

    mark_step("Enterprise validation")
    update("STEP 7 — Enterprise validation", 90)

    validation = validate_procedure(
        procedure,
        requirements,
        llm_client
    )

    ########################################################
    # STEP 8 — SAVE OUTPUTS
    ########################################################

    mark_step("Saving document")
    update("STEP 8 — Saving document", 97)

    output_path, output_path2 = save_procedure_docx(
        procedure,
        validation
    )

    ########################################################
    # FINALIZE TIMING
    ########################################################

    if step_start is not None and last_step is not None:
        step_timings[last_step] = round(
            time.time() - step_start,
            2
        )

    total_time = round(time.time() - start_time, 2)

    update(f"DONE — Completed in {total_time}s", 100)

    ########################################################
    # RETURN RESULTS
    ########################################################

    return {
        "procedure": procedure,
        "coverage": coverage,
        "validation": validation,
        "output_path": output_path,
        "output_path_validation": output_path2,

        # core artifacts
        "requirements": requirements,
        "procedure_template": procedure_template,

        # analytics
        "timings": step_timings,

        # section-rag transparency layer
        "retrieval_sources": retrieval_sources,
        "retrieval_provenance": retrieval_system.get("provenance_map", []),

        # system metadata
        "documents_loaded": len(sections),
        "total_requirements": len(requirements),
        "runtime_seconds": total_time
    }

In [27]:
############################################################
# EXAMPLE EXECUTION
############################################################
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

if __name__ == "__main__":

    result = run_workflow(
        user_request="Generate startup procedure for PX-400 pressure system",
        document_folder="documents",
        llm_client=llm_client,
        procedure_template=procedure_template,
        progress_callback=None
    )

    print(result["procedure"])


F:\Pytorch\venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
W0512 18:03:42.663000 3836 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
F:\Pytorch\venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


STEP 1 — Loading documents
Loaded sections: 3
STEP 2 — Extracting requirements
Extracted requirements: 0
STEP 3 — Deduplicating requirements
Unique requirements: 0
STEP 4 — Building retrieval system
STEP 5 — Generating procedure
STEP 6 — Requirement coverage validation
STEP 7 — Enterprise validation
STEP 8 — Saving document
DONE — Completed in 96.15s
# General Procedure

### . Objective

**. Objective**

• The objective of this startup procedure is to ensure safe and controlled operation of the PX-400 pressure system during initial power-up and initialization.

• This procedure outlines the steps required to start up the PX-400 pressure system, including checks and tests to verify its proper functioning.

• The purpose of this procedure is to prevent equipment damage, minimize potential hazards, and ensure compliance with relevant regulations and industry standards.

### . Scope

**Scope**

The startup procedure for the PX-400 pressure system is intended to ensure safe and proper opera

In [28]:
result["retrieval_provenance"][0]

IndexError: list index out of range

In [ ]:
for row in result["retrieval_sources"][:3]:
    print("=" * 50)
    print(f"Document     : {row['document']}")
    print(f"Section ID   : {row['section_id']}: {row['title']}")
    print()